# Notebook 06: Parallel Execution, Subgraphs, and Map-Reduce Patterns

## Learning Objectives

1. Implement parallel execution with the Send API
2. Create subgraphs for modularity
3. Build multi-agent architectures
4. Use map-reduce patterns
5. Implement deferred nodes (LangGraph 1.1.9 feature)

## Prerequisites

- Completed Notebooks 01-05
- Understanding of agents, tools, and state management

## 1. Introduction to Parallel Execution

### Why Parallel Execution?

Process multiple items simultaneously:
- 🚀 **Faster execution** - Don't wait for sequential processing
- 📊 **Batch operations** - Handle multiple requests at once
- 🔄 **Map-reduce workflows** - Process and aggregate results

### The Send API

```python
from langgraph.types import Send

# Instead of returning a node name, return Send objects:
return [Send("process_item", {"item": item}) for item in items]
```

This creates **multiple parallel executions** of the same node!

In [1]:
import os
from typing_extensions import TypedDict
from typing import Annotated
from langgraph.graph import StateGraph, START, END
from langgraph.types import Send
from langchain_nvidia_ai_endpoints import ChatNVIDIA
from langchain_core.messages import HumanMessage
from dotenv import load_dotenv
import operator

load_dotenv()
print("✅ Ready for parallel execution!")

✅ Ready for parallel execution!


/Users/aashishgarg/Downloads/Langraph/langraph-tutorials/.venv/lib/python3.12/site-packages/langgraph/cache/base/__init__.py:8: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


In [2]:
import time

def _invoke_with_backoff(runnable, input_data, config=None, max_retries=5):
    """Invoke any LangChain runnable with exponential backoff on 429 rate-limit errors."""
    delay = 5
    for attempt in range(max_retries):
        try:
            if config:
                return runnable.invoke(input_data, config)
            return runnable.invoke(input_data)
        except Exception as e:
            if "429" in str(e) and attempt < max_retries - 1:
                print(f"⏳ Rate limited — retrying in {delay}s (attempt {attempt+1}/{max_retries})...")
                time.sleep(delay)
                delay *= 2
            else:
                raise

print("✅ Backoff helper loaded")


✅ Backoff helper loaded


## 2. Example 1: Parallel Data Processor

Process multiple items in parallel using the Send API.

In [3]:
class BatchState(TypedDict):
    items: list  # Items to process
    results: Annotated[list, operator.add]  # Results accumulate

class ItemState(TypedDict):
    item: str  # Single item

def distribute_work(state: BatchState):
    """Distribute items for parallel processing."""
    print(f"📦 Distributing {len(state['items'])} items for processing")
    return [Send("process_item", {"item": item}) for item in state["items"]]

def process_item(state: ItemState) -> dict:
    """Process a single item."""
    item = state["item"]
    print(f"  ⚙️  Processing: {item}")
    result = f"Processed: {item.upper()}"
    return {"results": [result]}

# Build parallel processing graph
parallel_graph = StateGraph(BatchState)
parallel_graph.add_node("process_item", process_item)

# distribute_work goes here as the conditional edge function, not a node
parallel_graph.add_conditional_edges(START, distribute_work, ["process_item"])
parallel_graph.add_edge("process_item", END)

parallel_app = parallel_graph.compile()

# Test parallel processing
result = _invoke_with_backoff(parallel_app, {
    "items": ["apple", "banana", "cherry", "date"],
    "results": []
})

print("\n📊 Results:")
for r in result["results"]:
    print(f"  • {r}")

📦 Distributing 4 items for processing
  ⚙️  Processing: apple
  ⚙️  Processing: banana
  ⚙️  Processing: cherry
  ⚙️  Processing: date

📊 Results:
  • Processed: APPLE
  • Processed: BANANA
  • Processed: CHERRY
  • Processed: DATE


## 3. Subgraphs for Multi-Agent Systems

Subgraphs allow you to create modular, reusable workflows.

### Pattern:
```
Parent Graph:
  START → coordinator → [Subgraph 1, Subgraph 2] → aggregator → END

Subgraph 1:
  entry → process → exit

Subgraph 2:
  entry → analyze → exit
```

In [4]:
# Define a reusable subgraph for an agent
class AgentState(TypedDict):
    task: str
    result: str

def researcher_node(state: AgentState) -> dict:
    """Research agent subgraph."""
    task = state["task"]
    print(f"🔍 Researcher working on: {task}")
    return {"result": f"Research findings for: {task}"}

def analyzer_node(state: AgentState) -> dict:
    """Analyzer agent subgraph."""
    task = state["task"]
    print(f"📊 Analyzer working on: {task}")
    return {"result": f"Analysis of: {task}"}

# Build subgraphs
researcher_graph = StateGraph(AgentState)
researcher_graph.add_node("research", researcher_node)
researcher_graph.add_edge(START, "research")
researcher_graph.add_edge("research", END)
researcher_subgraph = researcher_graph.compile()

analyzer_graph = StateGraph(AgentState)
analyzer_graph.add_node("analyze", analyzer_node)
analyzer_graph.add_edge(START, "analyze")
analyzer_graph.add_edge("analyze", END)
analyzer_subgraph = analyzer_graph.compile()

print("✅ Subgraphs created!")

✅ Subgraphs created!


## 4. Advanced Project: Multi-Agent Research System

Build a system with multiple specialized agents working together:

- **Researcher**: Gathers information
- **Analyzer**: Evaluates credibility
- **Synthesizer**: Combines findings
- **Editor**: Formats output

In [5]:
from langgraph.graph.message import add_messages

class MultiAgentState(TypedDict):
    messages: Annotated[list, add_messages]
    research_results: list
    analysis_results: list
    final_report: str

# Create specialized LLMs for each agent
researcher_llm = ChatNVIDIA(model="meta/llama-3.3-70b-instruct", temperature=0.3)
analyzer_llm = ChatNVIDIA(model="meta/llama-3.3-70b-instruct", temperature=0)
synthesizer_llm = ChatNVIDIA(model="meta/llama-3.3-70b-instruct", temperature=0.5)

# coordinator becomes a pure edge function (no node registration)
def coordinator(state: MultiAgentState):
    query = state["messages"][-1].content
    print(f"🎯 Coordinator: Distributing task: {query}")
    return [
        Send("researcher", state),
        Send("analyzer", state)
    ]

def researcher(state: MultiAgentState) -> dict:
    query = state["messages"][-1].content
    print("🔬 Researcher: Gathering information...")
    response = _invoke_with_backoff(researcher_llm, [
        HumanMessage(content=f"Research this topic and provide key facts: {query}")
    ])
    return {"research_results": [response.content]}

def analyzer(state: MultiAgentState) -> dict:
    query = state["messages"][-1].content
    print("📊 Analyzer: Evaluating sources...")
    response = _invoke_with_backoff(analyzer_llm, [
        HumanMessage(content=f"Analyze the credibility and relevance of information about: {query}")
    ])
    return {"analysis_results": [response.content]}

def synthesizer(state: MultiAgentState) -> dict:
    """Synthesize all findings into a coherent report."""
    print("✍️  Synthesizer: Creating final report...")
    
    research = "\n".join(state.get("research_results", []))
    analysis = "\n".join(state.get("analysis_results", []))
    
    response = _invoke_with_backoff(synthesizer_llm, [
        HumanMessage(content=f"Synthesize these findings into a coherent report:\n\nResearch:\n{research}\n\nAnalysis:\n{analysis}")
    ])
    
    return {"final_report": response.content}

# Build multi-agent graph
multi_agent_builder = StateGraph(MultiAgentState)
# Remove: multi_agent_builder.add_node("coordinator", coordinator)
multi_agent_builder.add_node("researcher", researcher)
multi_agent_builder.add_node("analyzer", analyzer)
multi_agent_builder.add_node("synthesizer", synthesizer)

# coordinator goes here as the edge function from START
multi_agent_builder.add_conditional_edges(START, coordinator, ["researcher", "analyzer"])
multi_agent_builder.add_edge("researcher", "synthesizer")
multi_agent_builder.add_edge("analyzer", "synthesizer")
multi_agent_builder.add_edge("synthesizer", END)

multi_agent_system = multi_agent_builder.compile()

print("✅ Multi-agent research system ready!")

✅ Multi-agent research system ready!


In [6]:
# Test the multi-agent system
result = _invoke_with_backoff(multi_agent_system, {
    "messages": [HumanMessage(content="Explain the impact of quantum computing on cryptography")],
    "research_results": [],
    "analysis_results": [],
    "final_report": ""
})

print("\n" + "=" * 70)
print("📄 Final Research Report")
print("=" * 70)
print(result["final_report"])
print("=" * 70)

🎯 Coordinator: Distributing task: Explain the impact of quantum computing on cryptography
🔬 Researcher: Gathering information...
📊 Analyzer: Evaluating sources...
✍️  Synthesizer: Creating final report...

📄 Final Research Report
**Introduction to Quantum Computing and Cryptography: A Comprehensive Report**

Quantum computing is a revolutionary technology that leverages the principles of quantum mechanics to perform calculations and operations on data. This technology has the potential to significantly impact various fields, including cryptography, which is the practice of secure communication by transforming plaintext into unreadable ciphertext. Cryptography plays a crucial role in protecting sensitive information, and the advent of quantum computing poses a significant threat to traditional cryptography.

**Impact of Quantum Computing on Cryptography**

Quantum computers can potentially solve complex mathematical problems much faster than classical computers, compromising the securit

## 🆕 Deferred Nodes — Race-Condition-Free Aggregation

In map-reduce patterns, there's a race condition: the aggregator node might start before all parallel branches finish. **Deferred nodes** solve this by waiting for ALL upstream parallel paths to complete before executing.

```python
graph.add_node("synthesizer", synthesize_fn, defer=True)
```

Without `defer=True`, the synthesizer might run before all researchers finish. With it, LangGraph guarantees it only runs after ALL parallel Send() executions complete.

In [7]:
import operator
from typing import Annotated
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.types import Send

class ResearchState(TypedDict):
    topics: list[str]
    findings: Annotated[list[str], operator.add]  # accumulated from parallel branches
    final_report: str

def coordinator(state: ResearchState) -> list[Send]:
    """Distribute research topics in parallel."""
    print(f"📋 Coordinating research on {len(state['topics'])} topics")
    return [Send("research_topic", {"topics": [topic], "findings": [], "final_report": ""})
            for topic in state["topics"]]

def research_topic(state: ResearchState) -> dict:
    """Research a single topic (runs in parallel)."""
    topic = state["topics"][0]
    finding = f"Research finding for '{topic}': key insight discovered"
    print(f"🔬 Researching: {topic}")
    return {"findings": [finding]}

def synthesize_report(state: ResearchState) -> dict:
    """Synthesize all findings — runs AFTER all parallel branches complete."""
    print(f"\n📊 Synthesizing {len(state['findings'])} findings...")
    findings_text = "\n".join(f"• {f}" for f in state["findings"])
    report = "FINAL REPORT:\n" + findings_text
    return {"final_report": report}

# Build graph
builder = StateGraph(ResearchState)
# Remove: builder.add_node("coordinator", coordinator)
builder.add_node("research_topic", research_topic)
builder.add_node("synthesizer", synthesize_report, defer=True)

# coordinator is the conditional edge function from START, not a node
builder.add_conditional_edges(START, coordinator, ["research_topic"])
builder.add_edge("research_topic", "synthesizer")
builder.add_edge("synthesizer", END)

app = builder.compile()

result = _invoke_with_backoff(app, {
    "topics": ["AI Safety", "Multi-Agent Systems", "LangGraph Architecture"],
    "findings": [],
    "final_report": ""
})
print(result["final_report"])

📋 Coordinating research on 3 topics
🔬 Researching: AI Safety
🔬 Researching: Multi-Agent Systems
🔬 Researching: LangGraph Architecture

📊 Synthesizing 3 findings...
FINAL REPORT:
• Research finding for 'AI Safety': key insight discovered
• Research finding for 'Multi-Agent Systems': key insight discovered
• Research finding for 'LangGraph Architecture': key insight discovered


**Why `defer=True` matters**: Without it, if topic A finishes first, `synthesizer` might run with only 1 finding. With `defer=True`, LangGraph queues the synthesizer and only executes it when ALL parallel `research_topic` branches complete — guaranteed.

## 🆕 Subgraph-to-Parent Routing with Command

Subgraphs normally can't directly influence the parent graph's routing. With `Command(graph=Command.PARENT)`, a subgraph node can break out and route within the parent graph:

In [8]:
from langgraph.graph import StateGraph, START, END
from langgraph.types import Command
from typing_extensions import TypedDict

class ParentState(TypedDict):
    task: str
    escalated: bool
    result: str

class SubState(TypedDict):
    task: str
    escalated: bool
    result: str

# Subgraph with escape hatch
def process_in_subgraph(state: SubState) -> dict | Command:
    """Try to handle the task. If too complex, escalate to parent."""
    if "critical" in state["task"].lower():
        print("🚨 Subgraph: Task is critical — escalating to parent!")
        # Escape the subgraph and route in the PARENT graph
        return Command(
            goto="escalation_handler",   # node in the PARENT graph
            graph=Command.PARENT,        # ← escape to parent
            update={"escalated": True, "result": "Escalated to human review"}
        )
    
    result = f"✅ Subgraph handled: {state['task']}"
    return {"result": result}

# Build subgraph
sub_builder = StateGraph(SubState)
sub_builder.add_node("process", process_in_subgraph)
sub_builder.add_edge(START, "process")
sub_builder.add_edge("process", END)
subgraph = sub_builder.compile()

# Build parent graph
def start_task(state: ParentState) -> dict:
    print(f"🎯 Parent: Starting task '{state['task']}'" )
    return {}

def escalation_handler(state: ParentState) -> dict:
    print(f"👨‍💼 Parent: Handling escalation for '{state['task']}'" )
    return {"result": f"ESCALATED: '{state['task']}' sent to human review team"}

parent_builder = StateGraph(ParentState)
parent_builder.add_node("start", start_task)
parent_builder.add_node("subgraph_worker", subgraph)     # subgraph as node
parent_builder.add_node("escalation_handler", escalation_handler)

parent_builder.add_edge(START, "start")
parent_builder.add_edge("start", "subgraph_worker")
parent_builder.add_edge("subgraph_worker", END)
parent_builder.add_edge("escalation_handler", END)

parent_app = parent_builder.compile()

# Test 1: Normal task — handled by subgraph
print("=== Test 1: Normal task ===")
result = _invoke_with_backoff(parent_app, {"task": "process invoice", "escalated": False, "result": ""})
time.sleep(5)  # avoid 429 rate-limit
print(f"Result: {result['result']}")

# Test 2: Critical task — subgraph escalates to parent
print("=== Test 2: Critical task ===")
result = _invoke_with_backoff(parent_app, {"task": "CRITICAL system failure", "escalated": False, "result": ""})
print(f"Result: {result['result']}")

=== Test 1: Normal task ===
🎯 Parent: Starting task 'process invoice'
Result: ✅ Subgraph handled: process invoice
=== Test 2: Critical task ===
🎯 Parent: Starting task 'CRITICAL system failure'
🚨 Subgraph: Task is critical — escalating to parent!
👨‍💼 Parent: Handling escalation for 'CRITICAL system failure'
Result: ESCALATED: 'CRITICAL system failure' sent to human review team


## 5. Key Takeaways

### Concepts Mastered

1. **Send API**: Parallel execution of nodes
2. **Subgraphs**: Modular, reusable workflows
3. **Multi-agent systems**: Specialized agents working together
4. **Map-reduce patterns**: Distribute, process, aggregate
5. **Deferred nodes**: Wait for all parallel tasks to complete

### Best Practices

✅ **Use Send for independent tasks** - Each can run in parallel  
✅ **Design subgraphs for reusability** - DRY principle  
✅ **Coordinate agents carefully** - Clear responsibilities  
✅ **Use reducers for aggregation** - `operator.add`, etc.  

### What's Next?

In **Notebook 07**, you'll learn production-ready patterns including caching, hooks, error handling, and deployment!